In [ ]:
import os
import glob
import psycopg2
import pandas as pd
from psycopg2.extras import execute_values

# Database connection details
DB_NAME = "github_repos"
DB_USER = "postgres"
DB_PASSWORD = "Sphings@19"
DB_HOST = "localhost"
DB_PORT = "5432"

# Directory containing CSV files
RESULTS_FOLDER = "../results"

try:
    # Connect to PostgreSQL
    conn = psycopg2.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        host=DB_HOST,
        port=DB_PORT
    )
    cur = conn.cursor()

    # Get all CSV files in the results folder
    csv_files = glob.glob(os.path.join(RESULTS_FOLDER, "*.csv"))

    # Read and combine all CSV files into a single DataFrame
    all_data = pd.DataFrame()

    for file in csv_files:
        print(f"Reading {file}...")
        df = pd.read_csv(file)

        # Rename CSV columns to match the database
        df.rename(columns={
            'Hash': 'hash',
            'Project ID': 'project_id',
            'Version': 'version',
            'License': 'license',
            'Method Name': 'method_name',
            'File Location': 'file_location',
            'Function Code': 'function_code',
            'Repository URL': 'repository_url',
            'Query Project': 'query_project',
            'Violation': 'violation',
            'Source_project': 'Source_project',
            'Source_project_version':'Source_project_version'
        }, inplace=True)

        all_data = pd.concat([all_data, df], ignore_index=True)

    # Remove duplicates based on (hash, project_id)
    all_data.drop_duplicates(subset=['hash', 'project_id'], inplace=True)

    # Generate unique ID by combining hash and project_id
    all_data['_id'] = all_data['hash'].astype(str) + "_" + all_data['project_id'].astype(str)

    # Convert DataFrame to a list of tuples for batch insert
    records_to_insert = [
        (
            row['_id'], row['hash'], row['project_id'], row['version'], row['license'], row['method_name'],
            row['file_location'], row['function_code'], row['repository_url'], row['query_project'], row['violation'],
            row['Source_project'],row['Source_project_version']
        ) for _, row in all_data.iterrows()
    ]

    # Insert all records in bulk
    insert_query = """
     INSERT INTO repository_data (
        _id, hash, project_id, version, license, method_name,
        file_location, function_code, repository_url, query_project, violation, Source_project, Source_project_version
    ) VALUES %s
    ON CONFLICT (hash, project_id, version) DO NOTHING;
    """
    
    
    execute_values(cur, insert_query, records_to_insert)

    # Commit changes
    conn.commit()
    print(f"Inserted {len(records_to_insert)} new records successfully.")

except Exception as e:
    print("Error:", e)

finally:
    # Close connection
    if conn:
        cur.close()
        conn.close()

Reading ../results/microsoft_ApplicationInsights-aspnetcore_matches_2004025982_0.csv...
Reading ../results/microsoft_Vipr_matches_1515233357.csv...
Reading ../results/microsoft_DXUT_matches_1097314711_6.csv...
Reading ../results/microsoft_llvm_matches_342089363_14.csv...
Reading ../results/microsoft_dotnet-apiweb_matches_204900798_6.csv...
Reading ../results/microsoft_moodle-auth_oidc_matches_2370378995_0.csv...
Reading ../results/microsoft_moodle-local_o365_matches_1435090601_0.csv...
Reading ../results/google_turbine_matches_1914135041_0.csv...
Reading ../results/microsoft_azure-chat-for-java_matches_1308223533_1.csv...
Reading ../results/microsoft_cordova-plugin-ms-appinsights_matches_1994650060_17.csv...
Reading ../results/microsoft_SimpleTracks_matches_65278542_0.csv...
Reading ../results/microsoft_mail2bug_matches_3234032417_0.csv...
Reading ../results/microsoft_PTVS_matches_424764781_8.csv...
Reading ../results/microsoft_Sora_matches_27483849_0.csv...
Reading ../results/microsof